In [13]:
# ============================================================
# PHASE 0 — CELL 1
# Imports + Locked Configuration
# ============================================================

import json
import math
import sys
import time
from dataclasses import dataclass, asdict
from typing import Optional

import torch
import torch.nn as nn
import pennylane as qml

print("Phase 0 environment loaded")
print("=" * 50)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# LOCKED ARCHITECTURE
# ------------------------------------------------------------

N_QUBITS = 4
ENTANGLING_LAYERS = 2
D_MODEL = 64
D_FF = 128
N_HEADS = 4
N_TOKENS = 225          # 15 × 15 patch
TRAIN_BATCH_SIZE = 32
EPOCHS = 50
TOTAL_RUNS = 117

# Phase 0 benchmark placeholders
N_TRAIN = 8000
K_DIM = 20
N_CLASSES = 16

# Benchmark settings
WARMUP = 5
ITERS = 30

print("\nLocked configuration")
print("=" * 50)
print("Qubits:", N_QUBITS)
print("Entangling layers:", ENTANGLING_LAYERS)
print("d_model:", D_MODEL)
print("d_ff:", D_FF)
print("Attention heads:", N_HEADS)
print("Tokens per sample:", N_TOKENS)
print("Batch size:", TRAIN_BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Total runs:", TOTAL_RUNS)
print("K dimension:", K_DIM)
print("Classes:", N_CLASSES)
print("Warmup:", WARMUP)
print("Timed iterations:", ITERS)

Phase 0 environment loaded
Python: 3.10.19
PyTorch: 2.5.0+cu118
PennyLane: 0.42.3
CUDA available: True
GPU: NVIDIA RTX A4500

Locked configuration
Qubits: 4
Entangling layers: 2
d_model: 64
d_ff: 128
Attention heads: 4
Tokens per sample: 225
Batch size: 32
Epochs: 50
Total runs: 117
K dimension: 20
Classes: 16
Warmup: 5
Timed iterations: 30


In [14]:
# ============================================================
# PHASE 0 — CELL 2
# Quantum Circuit + Quantum Token Encoder
# ============================================================

def build_quantum_layer(device_name: str, diff_method: str, shots=None):
    """
    Build the locked quantum circuit:

    RY AngleEmbedding
        ↓
    StronglyEntanglingLayers (L=2)
        ↓
    <Z> expectation on each of 4 qubits
    """

    try:
        dev = qml.device(
            device_name,
            wires=N_QUBITS,
            shots=shots
        )
    except Exception as e:
        print(f"  [skip] Could not create device '{device_name}': {e}")
        return None

    weight_shape = qml.StronglyEntanglingLayers.shape(
        n_layers=ENTANGLING_LAYERS,
        n_wires=N_QUBITS
    )

    @qml.qnode(
        dev,
        interface="torch",
        diff_method=diff_method
    )
    def circuit(inputs, weights):

        qml.AngleEmbedding(
            inputs,
            wires=range(N_QUBITS),
            rotation="Y"
        )

        qml.StronglyEntanglingLayers(
            weights,
            wires=range(N_QUBITS)
        )

        return [
            qml.expval(qml.PauliZ(w))
            for w in range(N_QUBITS)
        ]

    try:
        layer = qml.qnn.TorchLayer(
            circuit,
            {"weights": weight_shape}
        )
    except Exception as e:
        print(f"  [skip] TorchLayer construction failed for "
              f"'{device_name}': {e}")
        return None

    return layer, dev


class QuantumTokenEncoder(nn.Module):
    """
    Classical projection
        k dimensions → 4 quantum angles

    Quantum circuit
        4 angles → 4 expectation values

    Classical projection
        4 → 64-dimensional token representation
    """

    def __init__(
        self,
        k_dim: int,
        device_name: str,
        diff_method: str
    ):
        super().__init__()

        self.angle_proj = nn.Linear(
            k_dim,
            N_QUBITS
        )

        result = build_quantum_layer(
            device_name,
            diff_method
        )

        if result is None:
            raise RuntimeError(
                f"Backend '{device_name}' unavailable."
            )

        self.q_layer, self.device_handle = result

        self.out_proj = nn.Linear(
            N_QUBITS,
            D_MODEL
        )

    def forward(self, tokens):
        # tokens: (batch, 225, k_dim)

        b, n, k = tokens.shape

        # Project PCA features → 4 quantum angles
        theta = math.pi * torch.tanh(
            self.angle_proj(tokens)
        )

        # Flatten batch + token dimensions
        # (32, 225, 4) → (7200, 4)
        flat = theta.reshape(
            b * n,
            N_QUBITS
        )

        # ONE batched quantum-layer invocation
        q_out = self.q_layer(flat)

        # Restore token structure
        q_out = q_out.reshape(
            b,
            n,
            N_QUBITS
        )

        # 4 quantum outputs → 64-dimensional token
        return self.out_proj(q_out)


print("Quantum encoder definitions loaded.")
print("Locked quantum circuit:")
print("  4 qubits")
print("  RY AngleEmbedding")
print("  2 StronglyEntanglingLayers")
print("  4 Pauli-Z expectation values")
print("  20 → 4 → quantum → 4 → 64")

Quantum encoder definitions loaded.
Locked quantum circuit:
  4 qubits
  RY AngleEmbedding
  2 StronglyEntanglingLayers
  4 Pauli-Z expectation values
  20 → 4 → quantum → 4 → 64


In [15]:
# ============================================================
# PHASE 0 — CELL 3
# QuantFormer Transformer Backbone
# ============================================================

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_tokens: int, d_model: int):
        super().__init__()

        pe = torch.zeros(
            n_tokens,
            d_model
        )

        pos = torch.arange(
            0,
            n_tokens,
            dtype=torch.float32
        ).unsqueeze(1)

        div = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):
        return x + self.pe


class QuantFormer(nn.Module):
    """
    Full QuantFormer used for realistic Phase 0 timing:

    Input
      ↓
    QuantumTokenEncoder
      20 → 4 → quantum → 4 → 64
      ↓
    Sinusoidal positional encoding
      ↓
    Transformer encoder layer
      ↓
    Mean pooling over 225 tokens
      ↓
    Classifier
    """

    def __init__(
        self,
        k_dim: int,
        n_classes: int,
        device_name: str,
        diff_method: str
    ):
        super().__init__()

        self.q_encoder = QuantumTokenEncoder(
            k_dim,
            device_name,
            diff_method
        )

        self.pos_enc = SinusoidalPositionalEncoding(
            N_TOKENS,
            D_MODEL
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            activation="relu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = encoder_layer

        self.classifier = nn.Linear(
            D_MODEL,
            n_classes
        )

    def forward(self, tokens):

        x = self.q_encoder(tokens)

        x = self.pos_enc(x)

        x = self.encoder(x)

        x = x.mean(dim=1)

        return self.classifier(x)


print("QuantFormer definition loaded.")
print()
print("Architecture:")
print("  Input:              [batch, 225, 20]")
print("  Quantum encoder:    [batch, 225, 64]")
print("  Positional encoding")
print("  Transformer:        4 heads, d_model=64, d_ff=128")
print("  Mean pooling:       225 → 1 token")
print("  Classifier:         64 → 16")

QuantFormer definition loaded.

Architecture:
  Input:              [batch, 225, 20]
  Quantum encoder:    [batch, 225, 64]
  Positional encoding
  Transformer:        4 heads, d_model=64, d_ff=128
  Mean pooling:       225 → 1 token
  Classifier:         64 → 16


In [16]:
# ============================================================
# PHASE 0 — CELL 4
# Official Batched-Execution Correctness Gate
# ============================================================

def verify_batched_execution(model, tokens):

    dev = model.q_encoder.device_handle

    call_count = {
        "execute": 0,
        "derivatives": 0,
        "jvp": 0,
        "vjp": 0,
    }

    # Save original execution methods
    original_execute = dev.execute
    original_derivatives = dev.execute_and_compute_derivatives
    original_jvp = dev.execute_and_compute_jvp
    original_vjp = dev.execute_and_compute_vjp

    # Wrappers
    def counting_execute(*args, **kwargs):
        call_count["execute"] += 1
        return original_execute(*args, **kwargs)

    def counting_derivatives(*args, **kwargs):
        call_count["derivatives"] += 1
        return original_derivatives(*args, **kwargs)

    def counting_jvp(*args, **kwargs):
        call_count["jvp"] += 1
        return original_jvp(*args, **kwargs)

    def counting_vjp(*args, **kwargs):
        call_count["vjp"] += 1
        return original_vjp(*args, **kwargs)

    # Install wrappers
    dev.execute = counting_execute
    dev.execute_and_compute_derivatives = counting_derivatives
    dev.execute_and_compute_jvp = counting_jvp
    dev.execute_and_compute_vjp = counting_vjp

    try:
        # Forward + backward
        output = model(tokens)
        loss = output.sum()
        loss.backward()

    finally:
        # Always restore original methods
        dev.execute = original_execute
        dev.execute_and_compute_derivatives = original_derivatives
        dev.execute_and_compute_jvp = original_jvp
        dev.execute_and_compute_vjp = original_vjp

    total_calls = sum(call_count.values())
    total_token_vectors = tokens.shape[0] * tokens.shape[1]

    # A per-token loop would require approximately
    # batch × tokens quantum invocations.
    per_token_calls = total_token_vectors

    # We require a dramatically smaller number of device executions.
    passed = total_calls < per_token_calls / 2

    print("=" * 60)
    print("BATCHED-EXECUTION CORRECTNESS GATE")
    print("=" * 60)

    print(f"Batch size:              {tokens.shape[0]}")
    print(f"Tokens per sample:       {tokens.shape[1]}")
    print(f"Total token vectors:     {total_token_vectors}")
    print()
    print("Intercepted executions:")
    print(f"  execute:               {call_count['execute']}")
    print(f"  derivatives:           {call_count['derivatives']}")
    print(f"  jvp:                   {call_count['jvp']}")
    print(f"  vjp:                   {call_count['vjp']}")
    print(f"  TOTAL:                 {total_calls}")
    print()
    print(f"Per-token loop would be: ~{per_token_calls} calls")
    print()

    if passed:
        print("RESULT: PASS")
        print("All tokens are processed through batched quantum execution.")
    else:
        print("RESULT: FAIL")
        print("Possible per-token execution detected.")

    print("=" * 60)

    return passed


# ------------------------------------------------------------
# Build a fresh model for the official gate
# ------------------------------------------------------------

gate_model = QuantFormer(
    k_dim=K_DIM,
    n_classes=N_CLASSES,
    device_name="lightning.qubit",
    diff_method="adjoint"
)

gate_model = gate_model.to("cpu")

gate_tokens = torch.randn(
    TRAIN_BATCH_SIZE,
    N_TOKENS,
    K_DIM,
    device="cpu",
    requires_grad=True
)

gate_passed = verify_batched_execution(
    gate_model,
    gate_tokens
)

BATCHED-EXECUTION CORRECTNESS GATE
Batch size:              32
Tokens per sample:       225
Total token vectors:     7200

Intercepted executions:
  execute:               0
  derivatives:           1
  jvp:                   0
  vjp:                   0
  TOTAL:                 1

Per-token loop would be: ~7200 calls

RESULT: PASS
All tokens are processed through batched quantum execution.


In [17]:
# ============================================================
# PHASE 0 — CELL 5
# Quantum Encoder Gradient Check
# ============================================================

def run_gradcheck(k_dim, device_name, diff_method):

    print(f"\nRunning gradcheck: {device_name} "
          f"(diff_method={diff_method})")

    # Gradcheck requires double precision
    torch.set_default_dtype(torch.float64)

    try:
        encoder = QuantumTokenEncoder(
            k_dim=k_dim,
            device_name=device_name,
            diff_method=diff_method
        ).double()

        # Small test case:
        # batch = 1
        # tokens = 3
        # features = K_DIM
        small_tokens = torch.randn(
            1,
            3,
            k_dim,
            dtype=torch.float64,
            requires_grad=True
        )

        def fn(t):
            return encoder(t).sum()

        print("Input shape:", tuple(small_tokens.shape))
        print("Running torch.autograd.gradcheck...")

        ok = torch.autograd.gradcheck(
            fn,
            (small_tokens,),
            eps=1e-4,
            atol=1e-3,
            rtol=1e-2
        )

        print()
        print(f"[{'PASS' if ok else 'FAIL'}] "
              f"gradcheck for '{device_name}'")

        return ok

    except Exception as e:

        print(f"[FAIL] gradcheck for '{device_name}'")
        print("Error:", e)

        return False

    finally:
        # Restore normal Phase 0 precision
        torch.set_default_dtype(torch.float32)


# ------------------------------------------------------------
# Run gradcheck on the backend that passed the batched gate
# ------------------------------------------------------------

gradcheck_passed = run_gradcheck(
    k_dim=K_DIM,
    device_name="lightning.qubit",
    diff_method="adjoint"
)


Running gradcheck: lightning.qubit (diff_method=adjoint)
Input shape: (1, 3, 20)
Running torch.autograd.gradcheck...

[PASS] gradcheck for 'lightning.qubit'


In [18]:
# ============================================================
# PHASE 0 — CELL 6
# Benchmark: default.qubit + backprop
# ============================================================

def benchmark_default_qubit(
    k_dim,
    n_classes,
    warmup,
    iters
):
    print("=" * 60)
    print("PHASE 0 BENCHMARK")
    print("Backend: default.qubit")
    print("Differentiation: backprop")
    print("=" * 60)

    device_name = "default.qubit"
    diff_method = "backprop"

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    try:
        model = QuantFormer(
            k_dim=k_dim,
            n_classes=n_classes,
            device_name=device_name,
            diff_method=diff_method
        )

    except Exception as e:
        print(f"[SKIP] Could not build model: {e}")
        return None

    # default.qubit is CPU
    classical_device = torch.device("cpu")
    model = model.to(classical_device)

    # --------------------------------------------------------
    # Optimizer + loss
    # --------------------------------------------------------

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Dummy training batch
    # --------------------------------------------------------

    dummy_tokens = torch.randn(
        TRAIN_BATCH_SIZE,
        N_TOKENS,
        k_dim,
        device=classical_device
    )

    dummy_labels = torch.randint(
        0,
        n_classes,
        (TRAIN_BATCH_SIZE,),
        device=classical_device
    )

    print(f"Input shape:  {tuple(dummy_tokens.shape)}")
    print(f"Labels shape: {tuple(dummy_labels.shape)}")
    print()

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    print(f"Warm-up: {warmup} iterations")

    model.train()

    for i in range(warmup):

        optimizer.zero_grad()

        output = model(dummy_tokens)

        loss = criterion(
            output,
            dummy_labels
        )

        loss.backward()

        optimizer.step()

        print(
            f"  warmup {i + 1}/{warmup} "
            f"| loss = {loss.item():.4f}"
        )

    # --------------------------------------------------------
    # Timed iterations
    # --------------------------------------------------------

    print()
    print(f"Timed benchmark: {iters} iterations")
    print("Starting timer...")

    start = time.perf_counter()

    for _ in range(iters):

        optimizer.zero_grad()

        output = model(dummy_tokens)

        loss = criterion(
            output,
            dummy_labels
        )

        loss.backward()

        optimizer.step()

    elapsed = time.perf_counter() - start

    seconds_per_batch = elapsed / iters

    print()
    print("-" * 60)
    print(f"Total timed time:       {elapsed:.4f} seconds")
    print(f"Iterations:              {iters}")
    print(f"Seconds per batch:       {seconds_per_batch:.4f}")
    print(f"Milliseconds per batch:  {seconds_per_batch * 1000:.2f}")
    print("-" * 60)

    return {
        "backend": device_name,
        "diff_method": diff_method,
        "seconds_per_batch": seconds_per_batch,
        "warmup": warmup,
        "iters": iters,
        "batch_size": TRAIN_BATCH_SIZE,
        "tokens_per_sample": N_TOKENS,
        "k_dim": k_dim,
    }


# ------------------------------------------------------------
# Run benchmark
# ------------------------------------------------------------

default_qubit_result = benchmark_default_qubit(
    k_dim=K_DIM,
    n_classes=N_CLASSES,
    warmup=WARMUP,
    iters=ITERS
)

PHASE 0 BENCHMARK
Backend: default.qubit
Differentiation: backprop
Input shape:  (32, 225, 20)
Labels shape: (32,)

Warm-up: 5 iterations
  warmup 1/5 | loss = 2.9319
  warmup 2/5 | loss = 2.6872
  warmup 3/5 | loss = 2.5145
  warmup 4/5 | loss = 2.4545
  warmup 5/5 | loss = 2.4891

Timed benchmark: 30 iterations
Starting timer...

------------------------------------------------------------
Total timed time:       2.8505 seconds
Iterations:              30
Seconds per batch:       0.0950
Milliseconds per batch:  95.02
------------------------------------------------------------


In [ ]:
# ============================================================
# PHASE 0 — CELL 7
# Benchmark: lightning.qubit + adjoint
# ============================================================

def benchmark_lightning_qubit(
    k_dim,
    n_classes,
    warmup,
    iters
):
    print("=" * 60)
    print("PHASE 0 BENCHMARK")
    print("Backend: lightning.qubit")
    print("Differentiation: adjoint")
    print("=" * 60)

    device_name = "lightning.qubit"
    diff_method = "adjoint"

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    try:
        model = QuantFormer(
            k_dim=k_dim,
            n_classes=n_classes,
            device_name=device_name,
            diff_method=diff_method
        )

    except Exception as e:
        print(f"[SKIP] Could not build model: {e}")
        return None

    # lightning.qubit runs on CPU
    classical_device = torch.device("cpu")
    model = model.to(classical_device)

    # --------------------------------------------------------
    # Optimizer + loss
    # --------------------------------------------------------

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Dummy training batch
    # --------------------------------------------------------

    dummy_tokens = torch.randn(
        TRAIN_BATCH_SIZE,
        N_TOKENS,
        k_dim,
        device=classical_device
    )

    dummy_labels = torch.randint(
        0,
        n_classes,
        (TRAIN_BATCH_SIZE,),
        device=classical_device
    )

    print(f"Input shape:  {tuple(dummy_tokens.shape)}")
    print(f"Labels shape: {tuple(dummy_labels.shape)}")
    print()

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    print(f"Warm-up: {warmup} iterations")

    model.train()

    for i in range(warmup):

        optimizer.zero_grad()

        output = model(dummy_tokens)

        loss = criterion(
            output,
            dummy_labels
        )

        loss.backward()

        optimizer.step()

        print(
            f"  warmup {i + 1}/{warmup} "
            f"| loss = {loss.item():.4f}"
        )

    # --------------------------------------------------------
    # Timed iterations
    # --------------------------------------------------------

    print()
    print(f"Timed benchmark: {iters} iterations")
    print("Starting timer...")

    start = time.perf_counter()

    for _ in range(iters):

        optimizer.zero_grad()

        output = model(dummy_tokens)

        loss = criterion(
            output,
            dummy_labels
        )

        loss.backward()

        optimizer.step()

    elapsed = time.perf_counter() - start

    seconds_per_batch = elapsed / iters

    print()
    print("-" * 60)
    print(f"Total timed time:       {elapsed:.4f} seconds")
    print(f"Iterations:              {iters}")
    print(f"Seconds per batch:       {seconds_per_batch:.4f}")
    print(f"Milliseconds per batch:  {seconds_per_batch * 1000:.2f}")
    print("-" * 60)

    return {
        "backend": device_name,
        "diff_method": diff_method,
        "seconds_per_batch": seconds_per_batch,
        "warmup": warmup,
        "iters": iters,
        "batch_size": TRAIN_BATCH_SIZE,
        "tokens_per_sample": N_TOKENS,
        "k_dim": k_dim,
    }


# ------------------------------------------------------------
# Run benchmark
# ------------------------------------------------------------

lightning_qubit_result = benchmark_lightning_qubit(
    k_dim=K_DIM,
    n_classes=N_CLASSES,
    warmup=WARMUP,
    iters=ITERS
)



#This benchmark was manually interrupted after 17 minutes on the timed 30-iteration test (warmup completed normally). This result is the basis for excluding lightning.qubit from the final backend selection — see cell 11

PHASE 0 BENCHMARK
Backend: lightning.qubit
Differentiation: adjoint
Input shape:  (32, 225, 20)
Labels shape: (32,)

Warm-up: 5 iterations
  warmup 1/5 | loss = 2.8191
  warmup 2/5 | loss = 2.7126
  warmup 3/5 | loss = 2.6675
  warmup 4/5 | loss = 2.6652
  warmup 5/5 | loss = 2.6691

Timed benchmark: 30 iterations
Starting timer...


KeyboardInterrupt: 

In [20]:
# ============================================================
# PHASE 0 — CELL 8
# Check / Benchmark: lightning.gpu + adjoint
# ============================================================

def benchmark_lightning_gpu(
    k_dim,
    n_classes,
    warmup,
    iters
):
    print("=" * 60)
    print("PHASE 0 BENCHMARK")
    print("Backend: lightning.gpu")
    print("Differentiation: adjoint")
    print("=" * 60)

    device_name = "lightning.gpu"
    diff_method = "adjoint"

    # --------------------------------------------------------
    # Try to create the model
    # --------------------------------------------------------
    try:
        model = QuantFormer(
            k_dim=k_dim,
            n_classes=n_classes,
            device_name=device_name,
            diff_method=diff_method
        )

    except Exception as e:
        print()
        print("[SKIP] lightning.gpu is not available in this environment.")
        print()
        print("Reason:")
        print(str(e))
        print()
        print("No installation or environment changes were made.")
        print("=" * 60)

        return {
            "backend": device_name,
            "diff_method": diff_method,
            "status": "unavailable",
            "error": str(e),
        }

    # --------------------------------------------------------
    # GPU model
    # --------------------------------------------------------
    if not torch.cuda.is_available():
        print("[SKIP] CUDA is not available.")
        return {
            "backend": device_name,
            "diff_method": diff_method,
            "status": "cuda_unavailable",
        }

    classical_device = torch.device("cuda")

    model = model.to(classical_device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    dummy_tokens = torch.randn(
        TRAIN_BATCH_SIZE,
        N_TOKENS,
        k_dim,
        device=classical_device
    )

    dummy_labels = torch.randint(
        0,
        n_classes,
        (TRAIN_BATCH_SIZE,),
        device=classical_device
    )

    print()
    print(f"Input shape:   {tuple(dummy_tokens.shape)}")
    print(f"Labels shape: {tuple(dummy_labels.shape)}")
    print()

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------
    print(f"Warm-up: {warmup} iterations")

    model.train()

    for i in range(warmup):

        optimizer.zero_grad()

        output = model(dummy_tokens)

        loss = criterion(
            output,
            dummy_labels
        )

        loss.backward()

        optimizer.step()

        torch.cuda.synchronize()

        print(
            f"  warmup {i + 1}/{warmup} "
            f"| loss = {loss.item():.4f}"
        )

    # --------------------------------------------------------
    # Timed benchmark
    # --------------------------------------------------------
    print()
    print(f"Timed benchmark: {iters} iterations")
    print("Starting timer...")

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(iters):

        optimizer.zero_grad()

        output = model(dummy_tokens)

        loss = criterion(
            output,
            dummy_labels
        )

        loss.backward()

        optimizer.step()

        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    seconds_per_batch = elapsed / iters

    print()
    print("-" * 60)
    print(f"Total timed time:       {elapsed:.4f} seconds")
    print(f"Iterations:              {iters}")
    print(f"Seconds per batch:       {seconds_per_batch:.4f}")
    print(f"Milliseconds per batch:  {seconds_per_batch * 1000:.2f}")
    print("-" * 60)

    return {
        "backend": device_name,
        "diff_method": diff_method,
        "status": "completed",
        "seconds_per_batch": seconds_per_batch,
        "warmup": warmup,
        "iters": iters,
        "batch_size": TRAIN_BATCH_SIZE,
        "tokens_per_sample": N_TOKENS,
        "k_dim": k_dim,
    }


lightning_gpu_result = benchmark_lightning_gpu(
    k_dim=K_DIM,
    n_classes=N_CLASSES,
    warmup=WARMUP,
    iters=ITERS
)

PHASE 0 BENCHMARK
Backend: lightning.gpu
Differentiation: adjoint
  [skip] Could not create device 'lightning.gpu': Device lightning.gpu does not exist. Make sure the required plugin is installed.

[SKIP] lightning.gpu is not available in this environment.

Reason:
Backend 'lightning.gpu' unavailable.

No installation or environment changes were made.


In [22]:
# ============================================================
# PHASE 0 — CELL 9
# Final Summary + 117-Run Extrapolation
# ============================================================

import json
import math

print("=" * 70)
print("PHASE 0 — BACKEND BENCHMARK & CORRECTNESS GATE")
print("=" * 70)

# ------------------------------------------------------------
# Locked workload
# ------------------------------------------------------------

batches_per_epoch = math.ceil(
    N_TRAIN / TRAIN_BATCH_SIZE
)

batches_per_run = batches_per_epoch * EPOCHS

print()
print("LOCKED WORKLOAD")
print("-" * 70)
print(f"Training samples:       {N_TRAIN}")
print(f"Batch size:             {TRAIN_BATCH_SIZE}")
print(f"Tokens per sample:     {N_TOKENS}")
print(f"K dimension:            {K_DIM}")
print(f"Classes:                {N_CLASSES}")
print(f"Epochs per run:         {EPOCHS}")
print(f"Runs in matrix:         {TOTAL_RUNS}")

print()
print(f"Batches per epoch:      {batches_per_epoch}")
print(f"Batches per run:        {batches_per_run}")

# ------------------------------------------------------------
# Backend results
# ------------------------------------------------------------

print()
print("BACKEND RESULTS")
print("-" * 70)

# Valid measured benchmark
default_result = {
    "backend": "default.qubit",
    "diff_method": "backprop",
    "status": "completed",
    "seconds_per_batch": default_qubit_result["seconds_per_batch"]
}

# Cell 7 was manually interrupted.
# No valid timing was produced.
lightning_result = {
    "backend": "lightning.qubit",
    "diff_method": "adjoint",
    "status": "aborted_excessively_slow",
    "seconds_per_batch": None
}

# Cell 8 confirmed GPU backend unavailable.
gpu_result = lightning_gpu_result

results = {
    "default_qubit": default_result,
    "lightning_qubit": lightning_result,
    "lightning_gpu": gpu_result,
}

for name, result in results.items():

    print(f"\n{name}")

    print(f"  status:          {result.get('status')}")
    print(f"  diff method:     {result.get('diff_method')}")

    if result.get("seconds_per_batch") is not None:

        print(
            f"  seconds/batch:   "
            f"{result['seconds_per_batch']:.4f}"
        )

        print(
            f"  milliseconds:    "
            f"{result['seconds_per_batch'] * 1000:.2f}"
        )

# ------------------------------------------------------------
# Correctness gates
# ------------------------------------------------------------

print()
print("CORRECTNESS GATES")
print("-" * 70)

print("Batched execution:")
print("  Backend:             lightning.qubit")
print("  Status:              PASS")
print("  Token vectors:       7,200")
print("  Intercepted calls:   1")

print()
print("Gradient check:")
print("  Backend:             lightning.qubit")
print("  Status:              PASS")

# ------------------------------------------------------------
# 117-RUN EXTRAPOLATION
# Based ONLY on the valid default.qubit measurement
# ------------------------------------------------------------

sec_batch = default_qubit_result["seconds_per_batch"]

seconds_per_run = (
    sec_batch * batches_per_run
)

hours_per_run = (
    seconds_per_run / 3600
)

total_seconds = (
    seconds_per_run * TOTAL_RUNS
)

total_hours = (
    total_seconds / 3600
)

total_days = (
    total_hours / 24
)

print()
print("117-RUN EXTRAPOLATION")
print("-" * 70)

print(
    f"Measured batch time:       "
    f"{sec_batch:.4f} seconds"
)

print(
    f"Batches per run:           "
    f"{batches_per_run}"
)

print(
    f"Estimated time per run:    "
    f"{seconds_per_run:.2f} seconds"
)

print(
    f"Estimated time per run:    "
    f"{hours_per_run:.4f} hours"
)

print(
    f"Estimated 117-run total:   "
    f"{total_hours:.2f} hours"
)

print(
    f"Estimated 117-run total:   "
    f"{total_days:.2f} days"
)

# ------------------------------------------------------------
# Save JSON
# ------------------------------------------------------------

phase0_results = {

    "configuration": {
        "n_train": N_TRAIN,
        "batch_size": TRAIN_BATCH_SIZE,
        "tokens_per_sample": N_TOKENS,
        "k_dim": K_DIM,
        "n_classes": N_CLASSES,
        "epochs": EPOCHS,
        "total_runs": TOTAL_RUNS,
        "warmup": WARMUP,
        "timed_iterations": ITERS,
        "n_qubits": N_QUBITS,
        "entangling_layers": N_LAYERS,
        "d_model": D_MODEL,
        "d_ff": D_FF,
        "attention_heads": N_HEADS,
    },

    "correctness": {

        "batched_execution": {
            "status": "PASS",
            "backend": "lightning.qubit",
            "diff_method": "adjoint",
            "batch_size": TRAIN_BATCH_SIZE,
            "tokens_per_sample": N_TOKENS,
            "total_token_vectors": (
                TRAIN_BATCH_SIZE * N_TOKENS
            ),
            "intercepted_executions": 1,
        },

        "gradcheck": {
            "status": "PASS",
            "backend": "lightning.qubit",
            "diff_method": "adjoint",
        }
    },

    "benchmarks": results,

    "extrapolation": {
        "batches_per_epoch": batches_per_epoch,
        "batches_per_run": batches_per_run,
        "seconds_per_batch": sec_batch,
        "seconds_per_run": seconds_per_run,
        "hours_per_run": hours_per_run,
        "total_hours_117_runs": total_hours,
        "total_days_117_runs": total_days,
    }
}

with open(
    "phase0_results.json",
    "w"
) as f:

    json.dump(
        phase0_results,
        f,
        indent=2
    )

print()
print("=" * 70)
print("PHASE 0 SUMMARY COMPLETE")
print("=" * 70)
print("Results saved to: phase0_results.json")

PHASE 0 — BACKEND BENCHMARK & CORRECTNESS GATE

LOCKED WORKLOAD
----------------------------------------------------------------------
Training samples:       8000
Batch size:             32
Tokens per sample:     225
K dimension:            20
Classes:                16
Epochs per run:         50
Runs in matrix:         117

Batches per epoch:      250
Batches per run:        12500

BACKEND RESULTS
----------------------------------------------------------------------

default_qubit
  status:          completed
  diff method:     backprop
  seconds/batch:   0.0950
  milliseconds:    95.02

lightning_qubit
  status:          aborted_excessively_slow
  diff method:     adjoint

lightning_gpu
  status:          unavailable
  diff method:     adjoint

CORRECTNESS GATES
----------------------------------------------------------------------
Batched execution:
  Backend:             lightning.qubit
  Status:              PASS
  Token vectors:       7,200
  Intercepted calls:   1

Gradient ch

NameError: name 'N_LAYERS' is not defined

In [23]:
# ------------------------------------------------------------
# Save JSON
# ------------------------------------------------------------

phase0_results = {

    "configuration": {
        "n_train": N_TRAIN,
        "batch_size": TRAIN_BATCH_SIZE,
        "tokens_per_sample": N_TOKENS,
        "k_dim": K_DIM,
        "n_classes": N_CLASSES,
        "epochs": EPOCHS,
        "total_runs": TOTAL_RUNS,
        "warmup": WARMUP,
        "timed_iterations": ITERS,
        "n_qubits": N_QUBITS,
        "entangling_layers": 2,
        "d_model": D_MODEL,
        "d_ff": D_FF,
        "attention_heads": N_HEADS,
    },

    "correctness": {

        "batched_execution": {
            "status": "PASS",
            "backend": "lightning.qubit",
            "diff_method": "adjoint",
            "batch_size": TRAIN_BATCH_SIZE,
            "tokens_per_sample": N_TOKENS,
            "total_token_vectors": (
                TRAIN_BATCH_SIZE * N_TOKENS
            ),
            "intercepted_executions": 1,
        },

        "gradcheck": {
            "status": "PASS",
            "backend": "lightning.qubit",
            "diff_method": "adjoint",
        }
    },

    "benchmarks": results,

    "extrapolation": {
        "batches_per_epoch": batches_per_epoch,
        "batches_per_run": batches_per_run,
        "seconds_per_batch": sec_batch,
        "seconds_per_run": seconds_per_run,
        "hours_per_run": hours_per_run,
        "total_hours_117_runs": total_hours,
        "total_days_117_runs": total_days,
    }
}

with open(
    "phase0_results.json",
    "w"
) as f:

    json.dump(
        phase0_results,
        f,
        indent=2
    )

print()
print("=" * 70)
print("PHASE 0 SUMMARY COMPLETE")
print("=" * 70)
print("Results saved to: phase0_results.json")


PHASE 0 SUMMARY COMPLETE
Results saved to: phase0_results.json


In [25]:
# CELL 10 — default.qubit + backprop, running on CUDA tensors
print("="*60)
print("TEST: default.qubit backprop mode on CUDA")
print("="*60)

cuda_ok = torch.cuda.is_available()
if not cuda_ok:
    print("[skip] CUDA not available to torch in this environment.")
else:
    device_variants_to_try = [
        {"name": "default.qubit", "kwargs": {"wires": N_QUBITS}},  # baseline attempt, move tensors manually
    ]

    try:
        model_cuda = QuantFormer(k_dim=20, n_classes=16,
                                  device_name="default.qubit", diff_method="backprop")
        model_cuda = model_cuda.to("cuda")

        # sanity: confirm weights actually moved
        first_param = next(model_cuda.parameters())
        print(f"model param device: {first_param.device}")

        dummy_tokens_cuda = torch.randn(TRAIN_BATCH_SIZE, N_TOKENS, 20, device="cuda")
        dummy_labels_cuda = torch.randint(0, 16, (TRAIN_BATCH_SIZE,), device="cuda")

        optimizer = torch.optim.Adam(model_cuda.parameters(), lr=2e-3, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()

        # confirm output actually lands on cuda (not silently falling back to cpu)
        out = model_cuda(dummy_tokens_cuda)
        print(f"output tensor device: {out.device}")
        if out.device.type != "cuda":
            print("[WARNING] Output is NOT on CUDA — backprop path fell back to CPU internally. "
                  "This test is invalid; do not report a CUDA timing.")
        else:
            # warmup
            for _ in range(5):
                optimizer.zero_grad()
                loss = criterion(model_cuda(dummy_tokens_cuda), dummy_labels_cuda)
                loss.backward()
                optimizer.step()
            torch.cuda.synchronize()

            # timed
            start = time.perf_counter()
            for _ in range(30):
                optimizer.zero_grad()
                loss = criterion(model_cuda(dummy_tokens_cuda), dummy_labels_cuda)
                loss.backward()
                optimizer.step()
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            ms_per_batch = (elapsed / 30) * 1000
            print(f"\ndefault.qubit + backprop on CUDA: {ms_per_batch:.2f} ms/batch")
            print(f"Compare to CPU baseline: 95.02 ms/batch")
            speedup = 95.02 / ms_per_batch
            print(f"Speedup factor: {speedup:.2f}x" if speedup > 1 else f"SLOWDOWN factor: {1/speedup:.2f}x")

    except Exception as e:
        print(f"[FAIL] CUDA backprop test raised: {e}")

TEST: default.qubit backprop mode on CUDA
model param device: cuda:0
output tensor device: cuda:0

default.qubit + backprop on CUDA: 59.10 ms/batch
Compare to CPU baseline: 95.02 ms/batch
Speedup factor: 1.61x


In [26]:
# CELL 11 — backend decision, in writing, before moving to Phase 1
print("="*60)
print("PHASE 0 — FINAL BACKEND DECISION")
print("="*60)
print("""
default.qubit + backprop (CPU): 95.02 ms/batch — MEASURED, VALID
lightning.qubit + adjoint:      DISQUALIFIED — 17+ min stall on 7,200-vector
                                 batch, consistent with per-tape iteration
                                 overhead inside the intercepted call; not a
                                 fair candidate for this workload shape.
lightning.gpu + adjoint:        UNAVAILABLE — plugin not installed, deferred
                                 due to CUDA13.2/cu118 mismatch risk.
default.qubit + backprop (CUDA): see Cell 10 result above.
""")
# manually fill in after reading Cell 10 output:
CHOSEN_BACKEND = "default.qubit"
CHOSEN_DIFF_METHOD = "backprop"
CHOSEN_DEVICE = "cuda"   # or "cpu", based on Cell 10 result
print(f"LOCKED BACKEND FOR PHASE 2+: {CHOSEN_BACKEND} / {CHOSEN_DIFF_METHOD} / {CHOSEN_DEVICE}")

PHASE 0 — FINAL BACKEND DECISION

default.qubit + backprop (CPU): 95.02 ms/batch — MEASURED, VALID
lightning.qubit + adjoint:      DISQUALIFIED — 17+ min stall on 7,200-vector
                                 batch, consistent with per-tape iteration
                                 overhead inside the intercepted call; not a
                                 fair candidate for this workload shape.
lightning.gpu + adjoint:        UNAVAILABLE — plugin not installed, deferred
                                 due to CUDA13.2/cu118 mismatch risk.
default.qubit + backprop (CUDA): see Cell 10 result above.

LOCKED BACKEND FOR PHASE 2+: default.qubit / backprop / cuda
